# CNN-LSTM (híbrido convolucional + LSTM): proceso de construcción

Cuaderno de proceso del modelo híbrido. Las celdas de código son lecturas ligeras (no entrenan
nada; solo redes diminutas con datos aleatorios) y avisan si falta algún archivo. Modelos
hermanos: [`01_xgboost_booster.ipynb`](01_xgboost_booster.ipynb) y
[`02_tcn.ipynb`](02_tcn.ipynb).

> **Estado de la validación.** El ajuste de hiperparámetros (Optuna) está terminado. La
> **validación de 24 cortes con `--folds` está pendiente** (en ejecución al escribir este
> cuaderno): las secciones 6 y 7 lo marcan y no contienen cifras de validación del CNN-LSTM.

In [ ]:
import os
import warnings
from datetime import date
from pathlib import Path

import polars as pl
import yaml

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Raiz del repositorio:", ROOT)


def leer_csv(ruta):
    try:
        return pl.read_csv(ruta)
    except FileNotFoundError:
        print(f"No existe {ruta}. Ver la seccion 'Como reproducir'.")
    except Exception as error:
        print(f"No se pudo leer {ruta}: {error}")


def leer_yaml(ruta):
    with open(ruta, encoding="utf-8") as archivo:
        return yaml.safe_load(archivo)

## 1. Objetivo y criterio de selección

Pronosticar el nivel diario del Orinoco en **Ciudad Bolívar** y **Palúa** a 1 a 14 días,
reportando en particular los **días 7 y 14** (instrucción del profesor). Este cuaderno cuenta
cómo se construyó el modelo **CNN-LSTM**: qué se decidió, por qué y qué se descartó. Es el modelo híbrido que pidió el equipo: la parte convolucional del TCN más una LSTM que añade memoria recurrente.

**Criterio de selección** (fijado antes de ver resultados):

1. `rel_mae < 1` en **cada** estación, donde `rel_mae` = MAE del modelo / MAE de la persistencia
   (Naive: repetir el último nivel observado) en los mismos cortes.
2. Entre los que cumplen, el menor MAE medio entre las dos estaciones.

**Por qué `rel_mae` y no MASE.** Resuelve el problema de comparar contra una referencia justa.
Por dentro, divide dos errores calculados sobre las mismas ventanas. Se descartó el MASE con
escala del historial de entrenamiento: esa escala puede dar valores mayores que 1 a la propia
persistencia si el período evaluado fue más difícil que el de entrenamiento, y entonces el
umbral "menor que 1" deja de significar algo. `rel_mae` sería la decisión equivocada si la
persistencia fuera casi perfecta (MAE cercano a cero), porque el cociente se vuelve inestable.

## 2. Datos

**Fuente:** `data/raw/DATOS RIO ORINOCO.xlsx`, hoja `Entrenar` (niveles diarios de Ayacucho,
Caicara, Ciudad Bolívar y Palúa), usando **solo 2009-01-01 a 2024-12-31** (período válido
según el profesor): 5 844 días por estación. Los archivos de `data/raw/` no se modifican.

**Qué ve el modelo.** El frame que reciben todos los runners es el parquet de
`TimeSeriesDataset` (`data/interim/orinoco_dataset.parquet`) limpiado con `clean_history`
([`features/dataset.py`](../src/orinoco_forecast/features/dataset.py)). Trae las **cuatro
estaciones como series** (`unique_id`), el nivel `y`, lluvia de NASA POWER (`rain_*`), ONI y
columnas de rezagos ya calculadas. El CNN-LSTM toma `y` y las covariables que dejan `drop_covariates` (`[rain_, oni]` se ocultan) de los últimos `input_size` días hasta el corte, y es una sola red para las cuatro series.

### Limpieza (`clean_history`) y por qué

| Paso | Qué resuelve | Cómo funciona | Alternativa descartada | Cuándo sería un error |
|---|---|---|---|---|
| Recortar a 2009-2024 | Datos fuera del período no son válidos | Filtra `ds` con `model_frame.start/end` de `configs/data.yaml` | Usar la serie desde 1974: más historia, pero el profesor la declaró no válida | Si se confirma que el período anterior sí es válido |
| Anular rachas planas (`suspect_flat`, 10 días o más iguales) | Un sensor detenido repite el mismo valor y enseñaría persistencia falsa | `label_quality` etiqueta y `clean_series` las pone en nulo | Dejarlas: contamina el aprendizaje | Si una racha plana fuera real (río en estiaje extremo) |
| Interpolar huecos de hasta 31 días | Los modelos no admiten nulos | Interpolación lineal por fecha (`max_gap_days: 31`); marca `imputed = True` | Rellenar con la media o con el último valor: distorsiona la dinámica | Si el hueco cubre una crecida: la línea recta la borra |
| Descartar los primeros 60 días de 2009 | Las columnas de rezagos (hasta `oni_lag_60`) aún no existen | `drop_nulls()` tras limpiar; la serie efectiva empieza el 2009-03-02 | Rellenar rezagos con ceros: inventa datos | Nunca es un error mientras no se invente nada |

En 2009-2024: 3 valores `missing` en Palúa (29 de febrero de 2012, 2016 y 2020), rachas planas
de Ayacucho (2019-03-07 a 2019-03-18) y Ciudad Bolívar (2020-03-24 a 2020-04-11), y un
`suspect_jump` de Palúa (2021-10-29) que se etiqueta pero no se modifica: **34 filas
interpoladas**. La última interpolada (2020-04-11) es anterior a todos los cortes de
validación, así que la interpolación no usa datos posteriores a un corte.

In [ ]:
try:
    from orinoco_forecast.evaluation.execution import load_history

    historial = load_history()
    resumen = (
        historial.group_by("unique_id")
        .agg(
            pl.len().alias("dias"),
            pl.col("ds").min().alias("desde"),
            pl.col("ds").max().alias("hasta"),
            pl.col("imputed").sum().alias("dias_imputados"),
        )
        .sort("unique_id")
    )
    print(resumen)
    print("Columnas que reciben los runners:", len(historial.columns))
    print(historial.columns)
except FileNotFoundError:
    print(
        "Falta data/interim/orinoco_dataset.parquet. Generalo con:\n"
        "  uv run python scripts/run_model.py booster --write-dataset"
    )
except Exception as error:
    print("No se pudo cargar el historial:", error)

## 3. Partición de los datos

| Bloque | Qué es |
|---|---|
| Entrenamiento | Todo `ds <= cutoff`; **crece** con cada corte. Cada corte reajusta el modelo. |
| Validación | Los 14 días siguientes a cada corte: **24 cortes** de 2021-01-26 a 2022-12-17, uno cada 30 días. |
| Test final | `ds >= 2023-01-01`. **Nunca se usó** para decidir nada. |

**Qué problema resuelve.** Una serie de tiempo no se puede barajar ni repartir al azar: el
modelo no debe ver el futuro de lo que predice. **Cómo funciona:** `walk_forward_splits`
([`data/splits.py`](../src/orinoco_forecast/data/splits.py)) genera los cortes hacia atrás
desde `test_start - (horizonte + 1)` y falla si una ventana invade el test. **Alternativa
descartada:** un reparto fijo tipo 70/15/15, que mide un solo período y no dice si el modelo
es consistente; con 24 cortes se mide la consistencia. **Sería un error** si hubiera pocos
datos (aquí 24 ventanas de 14 días son una muestra razonable, no grande).

El test se evalúa **una sola vez**, con el modelo que elija el equipo. Si se mira antes para
decidir, la predicción deja de ser genuina.

In [ ]:
try:
    from orinoco_forecast.data.splits import walk_forward_splits

    cfg = leer_yaml("configs/backtest.yaml")
    test_start = date.fromisoformat(str(cfg["test_start"]))
    fechas = historial.filter(pl.col("unique_id") == "ciudad_bolivar").sort("ds")["ds"].to_list()
    cortes = walk_forward_splits(
        fechas,
        test_start=test_start,
        horizon=cfg["horizon"],
        step_size=cfg["step_size"],
        n_windows=cfg["n_windows"],
    )
    print(len(cortes), "cortes:", cortes[0].cutoff, "a", cortes[-1].cutoff)
    print("La ultima validacion termina el", cortes[-1].validation_end, "| el test empieza el", test_start)
    assert cortes[-1].validation_end < test_start
    solapadas = [b.validation_start <= a.validation_end for a, b in zip(cortes, cortes[1:])]
    print("Ventanas de validacion solapadas:", any(solapadas))
except NameError:
    print("Ejecuta antes la celda de datos (falta el historial).")
except Exception as error:
    print("No se pudo calcular la particion:", error)

## 4. Diseño del modelo

Código: [`models/neural/cnn_lstm.py`](../src/orinoco_forecast/models/neural/cnn_lstm.py)
(`CnnLstm` y `CnnLstmRunner`) y configuración
[`configs/models/cnn_lstm.yaml`](../configs/models/cnn_lstm.yaml).

**Qué significa "híbrido".** Hay dos cosas con ese nombre. **CNN-LSTM**: convoluciones 1D
extraen características de la ventana y una LSTM las procesa en orden (dos bloques en serie; es
lo que se implementó). **ConvLSTM** (Shi et al., 2015): las puertas de la LSTM usan
convoluciones sobre una cuadrícula espacial; se descartó porque aquí no hay cuadrícula, solo
cuatro estaciones a lo largo de un río.

```text
ventana (112 días x variables)  ->  Conv1d causal + ReLU  ->  LSTM (2 capas)  ->  último estado  ->  capa lineal  ->  14 cambios diarios
```

| Decisión | Qué problema resuelve | Cómo funciona por dentro | Alternativa descartada y por qué | Cuándo sería un error |
|---|---|---|---|---|
| **PyTorch propio** | `neuralforecast` 3.3.0 tiene `TCN` y `LSTM` pero **ningún híbrido** (verificado en `neuralforecast.models`) | Un `nn.Module` dentro del contrato `Runner` (`fit`/`predict`) | Un híbrido con `neuralforecast`: no existe. Sale de la regla del equipo "Nixtla para todos los modelos", por eso es una decisión que el equipo debe aceptar | Si el equipo exigiera cero código de entrenamiento propio |
| **Convolución causal** (relleno solo a la izquierda) | Que la salida de un día no vea días posteriores | `ConstantPad1d((k-1, 0))` antes de `Conv1d` | Relleno simétrico: filtraría información del futuro dentro de la ventana | Nunca en pronóstico |
| **LSTM sobre las características convolucionales**, último estado oculto a la capa lineal | Añadir memoria recurrente que el TCN no tiene | `nn.LSTM(batch_first=True)` con 2 capas y `dropout` entre ellas | Solo convolución (TCN): el contraste de este proyecto | Si la serie no tuviera dependencia de largo plazo |
| **Cabeza lineal a 14 pasos** (salida multihorizonte directa) | Predecir todo el horizonte de una vez | `Linear(lstm_hidden_size, 14)` sobre los cambios escalados | Decodificador recursivo: acumula error | Si el horizonte fuera mucho mayor |
| **Objetivo en diferencias** + `level_y` | Que "cambio cero" equivalga a la persistencia | `y_t - y_(t-1)`; el nivel se reconstruye con `y(corte) + cumsum` (`reconstruct_levels`) | Predecir niveles: la red debe reproducir el valor exacto | Si no hubiera inercia |
| **Escalado por ventana** (mediana y rango intercuartil de su propia entrada) | Que las crecidas no distorsionen la escala y que no se ajuste ningún escalador con datos posteriores al corte | `robust_scale` + piso `MIN_SCALE = 1e-3` para el IQR, así una ventana plana no explota | Escalar con todo el entrenamiento: filtraría estadísticas futuras al entrenar | Si las crecidas fueran lo único que importa |
| **Pérdida L1** | Coincidir con el MAE del criterio | `L1` sobre los objetivos escalados | MSE: pondera distinto las crecidas | Si importara el error cuadrático |
| **Parada temprana sin fuga** | Evitar sobreajuste | Las ventanas cuyo objetivo cae entero en los últimos 365 días validan; **las que cruzan la frontera se descartan** | Validar con ventanas que cruzan: el objetivo de entrenamiento y el de validación compartirían días | Si hubiera muy pocos datos |
| **Barajar las ventanas de entrenamiento** con un generador con semilla | Evitar que el orden de las ventanas sesgue la optimización | `torch.Generator` con la semilla de cada red | No barajar: la skill del proyecto prohíbe barajar *observaciones*; aquí se barajan **ventanas ya construidas**, la partición temporal sigue cronológica | Nunca mientras ninguna ventana mire más allá del corte |
| **3 semillas** (`CNNLSTM_s1..s3`) | Pesos iniciales aleatorios | Una red por semilla sobre las mismas ventanas; no se promedian | Una semilla: no distingue mejora de suerte | Si el costo fuera prohibitivo |
| **`drop_covariates: [rain_, oni]`** | Mismo criterio que el TCN: la lluvia y el ONI lo perjudicaron | Oculta las columnas con esos prefijos; sin covariables futuras (el parquet no trae Fourier) | Pasarle todo | Si una red recurrente aprovechara la lluvia mejor que la convolucional (no se probó) |

In [ ]:
try:
    import neuralforecast
    import neuralforecast.models as modelos

    nombres = sorted(n for n in dir(modelos) if n[:1].isupper())
    print("neuralforecast", neuralforecast.__version__)
    print("Existen TCN y LSTM:", "TCN" in nombres, "LSTM" in nombres)
    print("Algun modelo hibrido CNN-LSTM:", [n for n in nombres if "CNN" in n.upper() or "CONVLSTM" in n.upper()])
except Exception as error:
    print("No se pudo comprobar neuralforecast:", error)

In [ ]:
try:
    import numpy as np
    import torch

    from orinoco_forecast.models.neural.cnn_lstm import (
        CnnLstm, make_windows, reconstruct_levels, robust_scale,
    )

    torch.set_num_threads(1)
    torch.manual_seed(0)
    cfg = leer_yaml("configs/models/cnn_lstm.yaml")
    n_variables = 6  # numero de entradas de juguete
    red = CnnLstm(
        n_features=n_variables,
        horizon=14,
        conv_channels=cfg["conv_channels"],
        kernel_size=cfg["kernel_size"],
        conv_layers=cfg["conv_layers"],
        lstm_hidden_size=cfg["lstm_hidden_size"],
        lstm_layers=cfg["lstm_layers"],
        dropout=cfg["dropout"],
    ).eval()
    print("Parametros entrenables:", sum(p.numel() for p in red.parameters()))

    lote = torch.randn(4, cfg["input_size"], n_variables)
    print("Entrada (lote, dias, variables):", tuple(lote.shape), "-> salida:", tuple(red(lote).shape))

    # Causalidad: cambiar el ultimo dia no altera las salidas convolucionales anteriores
    modificado = lote.clone()
    modificado[:, -1, :] += 100.0
    antes = red.conv(lote.transpose(1, 2))
    despues = red.conv(modificado.transpose(1, 2))
    print("Salidas convolucionales anteriores al ultimo dia iguales:", bool(torch.allclose(antes[..., :-1], despues[..., :-1])))

    # Ventanas, escalado por ventana y reconstruccion de niveles
    serie = np.random.default_rng(0).normal(size=(300, 3)).astype("float32")
    x, y, finales = make_windows(serie, cfg["input_size"], 14)
    print("make_windows ->", x.shape, y.shape, "primer objetivo termina en la fila", int(finales[0]))
    escalada, mediana, iqr = robust_scale(torch.tensor(x[:2]))
    print("robust_scale: mediana", tuple(mediana.shape), "IQR", tuple(iqr.shape))
    cambios = torch.zeros(1, 14)
    print("Cambios cero reconstruyen la persistencia:", reconstruct_levels(cambios, torch.tensor([7.5]))[0, :3].tolist())
except Exception as error:
    print("No se pudo ejecutar la demostracion del modulo:", error)

In [ ]:
try:
    cfg = leer_yaml("configs/models/cnn_lstm.yaml")
    for clave in ["target_transform", "input_size", "conv_channels", "kernel_size", "conv_layers",
                  "lstm_hidden_size", "lstm_layers", "dropout", "learning_rate", "batch_size",
                  "max_epochs", "val_size", "early_stop_patience", "drop_covariates", "seeds"]:
        print(f"{clave}: {cfg[clave]}")
except Exception as error:
    print("No se pudo leer configs/models/cnn_lstm.yaml:", error)

## 5. Ajuste de hiperparámetros con Optuna

Mismo método que los otros dos modelos (TPE, **20 pruebas completas**, objetivo `rel_mae`
medio, 12 ventanas de ajuste en 2020: cortes del 2020-05-04 al 2021-01-12 cada 23 días). Lo
específico del CNN-LSTM:

| Decisión | Qué problema resuelve | Cómo funciona | Alternativa descartada | Cuándo sería un error |
|---|---|---|---|---|
| **Ajustar una vez y predecir en los 12 cortes** (`refit` falso) | Un entrenamiento por prueba en vez de 12 | Se ajusta con `ds <= 2020-05-04` y el modelo predice en cada corte | Reajustar en cada corte (como `--folds`): 12 veces más caro | Si el modelo cambiara mucho al añadir datos (la validación sí reajusta) |
| **Una prueba = las 3 semillas** | Medir el ruido de inicialización | `rel_mae` promediado entre semillas | Una semilla por prueba (como el TCN): más rápido, más ruidoso | Si el tiempo fuera crítico |
| **Podar LSTM de 128 unidades con 2 capas** | La prueba 17 pasó de 30 minutos en CPU y se detuvo | `TrialPruned` en el muestreo de parámetros | Dejarla correr o recortar el espacio de entrada: ambas obligaban a repetir el estudio | **Hay un matiz honesto**: la poda se aplica a esa combinación para *todas* las ventanas (28, 56, 112), aunque la lentitud se midió con la de 112 días |

Espacio buscado: `input_size` 28, 56 o 112; `conv_channels` 16, 32 o 64; `kernel_size` 2, 3 o 5;
`conv_layers` 1 a 3; `lstm_hidden_size` 32, 64 o 128; `lstm_layers` 1 o 2; `dropout` 0 a 0,3;
`learning_rate` 3e-4 a 1e-2 (logarítmico); `batch_size` 128 o 256; 30 épocas máximas con
paciencia 5.

**Una prueba quedó abierta.** La prueba 17 se detuvo a mano y figura como `RUNNING` en el CSV (21
filas); no cuenta entre las 20 completas.

In [ ]:
pruebas = leer_csv("reports/tuning/cnn_lstm_trials.csv")
if pruebas is not None:
    print(pruebas["state"].value_counts())
    completas = pruebas.filter(pl.col("state") == "COMPLETE").sort("value")
    columnas = [c for c in ["number", "value", "rel_mae_ciudad_bolivar", "rel_mae_palua",
                            "params_input_size", "params_conv_layers", "params_lstm_hidden_size",
                            "params_lstm_layers", "params_learning_rate", "duration_s"] if c in completas.columns]
    with pl.Config(tbl_cols=-1, tbl_width_chars=220, float_precision=4):
        print(completas.select(columnas).head(5))
    mejor = completas.row(0, named=True)
    print("Mejor prueba:", mejor["number"], "| rel_mae medio", round(mejor["value"], 4),
          "| CB", round(mejor["rel_mae_ciudad_bolivar"], 3), "| Palua", round(mejor["rel_mae_palua"], 3))
    print("Duracion de las pruebas completas: de", round(completas["duration_s"].min() / 60, 1),
          "a", round(completas["duration_s"].max() / 60, 1), "minutos")

### Qué cambió el ajuste respecto a los valores por defecto

| Parámetro | Por defecto (sin ajustar) | Ajustado |
|---|---|---|
| `input_size` | 56 | 112 |
| `conv_layers` | 2 | 1 |
| `lstm_hidden_size` | 64 | 32 |
| `lstm_layers` | 1 | 2 |
| `dropout` | 0,0 | 0,0378 |
| `learning_rate` | 0,002 | 0,000439 |

`conv_channels` (32), `kernel_size` (3) y `batch_size` (256) no cambiaron. El ajuste prefirió una
ventana larga (112 días, como el TCN), una red pequeña y una tasa de aprendizaje baja.

## 6. Comparación con los otros modelos

### Con la misma medición de ajuste (12 ventanas de 2020, `rel_mae` medio)

Esta es la única comparación hoy en igualdad de condiciones entre los tres modelos. Se lee
directamente de los tres CSV de `reports/tuning/`:

In [ ]:
filas = []
for modelo, archivo in [("XGBoost (booster)", "xgboost"), ("TCN", "tcn"), ("CNN-LSTM", "cnn_lstm")]:
    tabla = leer_csv(f"reports/tuning/{archivo}_trials.csv")
    if tabla is None:
        continue
    mejor = tabla.filter(pl.col("state") == "COMPLETE").sort("value").row(0, named=True)
    filas.append({
        "modelo": modelo,
        "mejor_prueba": mejor["number"],
        "rel_mae_medio": round(mejor["value"], 3),
        "ciudad_bolivar": round(mejor["rel_mae_ciudad_bolivar"], 3),
        "palua": round(mejor["rel_mae_palua"], 3),
    })
if filas:
    with pl.Config(tbl_cols=-1, tbl_width_chars=200):
        print(pl.DataFrame(filas))

### Validación de 24 cortes (la que cuenta para elegir)

| Modelo | Ciudad Bolívar, días 1 a 14 (MAE / `rel_mae`) | Palúa, días 1 a 14 (MAE / `rel_mae`) | Palúa, día 7 (`rel_mae`) |
|---|---|---|---|
| Naive | 0,570 / 1,00 | 0,488 / 1,00 | 1,00 |
| XGBoost (booster) | 0,278 / 0,49 | 0,341 / 0,70 | 0,83 |
| TCN (media de 3 semillas) | 0,294 / 0,52 | 0,383 / 0,78 | 0,93 |
| **CNN-LSTM** | **pendiente** | **pendiente** | **pendiente** |

**La validación de 24 cortes del CNN-LSTM está pendiente.** No se cita ninguna cifra de
validación del híbrido en este cuaderno: la celda siguiente la leerá de
`data/predictions/cnn_lstm/backtest/naive/comparison.parquet` cuando exista.

**Lo que sí se puede decir con lo medido (ajuste, no validación):** el CNN-LSTM va por detrás del
XGBoost y del TCN (0,700 contra 0,490 y 0,606), con la diferencia concentrada en **Palúa**
(0,903, solo un 10 % mejor que Naive). En Ciudad Bolívar (0,497) es competitivo: mejor que el
TCN en el ajuste (0,669). Son medias de una muestra de 12 ventanas y no sustituyen a la
validación, que puede cambiar el orden.

In [ ]:
def comparacion(runner):
    ruta = ROOT / f"data/predictions/{runner}/backtest/naive/comparison.parquet"
    try:
        tabla = pl.read_parquet(ruta)
    except FileNotFoundError:
        print(f"No existe {ruta}.\nGeneralo con: uv run python scripts/run_model.py {runner} --against naive --folds")
        return None
    except Exception as error:
        print(f"No se pudo leer {ruta}: {error}")
        return None
    try:
        columnas = ["unique_id", "model", "horizon", "mean_model", "mean_baseline",
                    "mean_improvement_pct", "win_rate", "sign_test_p"]
        return (
            tabla.filter((pl.col("metric") == "mae") & pl.col("horizon").cast(pl.String).is_in(["7", "14", "all"]))
            .select(columnas)
            .sort("unique_id", "model", "horizon")
        )
    except Exception as error:
        print("El archivo existe pero no tiene el formato esperado:", error)
        return tabla


resultado = comparacion("cnn_lstm")
if resultado is not None:
    with pl.Config(tbl_cols=-1, tbl_width_chars=200, float_precision=4):
        print(resultado)

## 7. Limitaciones y siguientes pasos

- **Reajuste por corte vs Optuna.** `--folds` reentrena en cada corte; Optuna eligió los
  hiperparámetros con un ajuste único y 12 ventanas de 2020. Los parámetros siguen siendo
  válidos, pero la medición no es idéntica.
- **Sin Fourier.** El parquet no trae `sin_*`/`cos_*` (`calendar_terms: 0`), así que el modelo
  no ve el ciclo anual de forma explícita. No se midió cuánto cambiaría.
- **Entrenan con las cuatro estaciones.** `execute` pasa las cuatro series y filtra la
  predicción a las dos objetivo. No se probó entrenar solo con las dos objetivo.
- **NSE negativo.** Cada ventana dura 14 días y su varianza interna es pequeña, así que el
  NSE castiga cualquier error (también a Naive). Se interpreta `rel_mae`.
- **Una validación de 24 cortes es una muestra moderada.** Los resultados se leen junto con el
  test de signos, no solo con el promedio.
- **Test final sin tocar.** Estos resultados son de validación y pueden ser algo optimistas.
- **Sin datos de la represa de Guri** (regula el Caroní, cerca de Palúa).
- **La validación de 24 cortes del CNN-LSTM está pendiente.** Con la configuración ajustada (ventana de
  112 días) es lenta en CPU: la corrida en curso lleva más de dos horas.
- **Estimación de ajuste con 12 ventanas**, un solo juego de semillas; Palúa (0,903) es el punto débil.
- **PyTorch propio**: no entra en la regla "Nixtla para todos los modelos"; el equipo debe aceptarlo.
- **Poda de LSTM 128 x 2 capas para todas las ventanas** (aunque la lentitud se midió con 112 días).
- **Determinismo en GPU:** el runner elige CUDA si está disponible pero no activa las opciones de
  determinismo de cuDNN, así que dos corridas en GPU pueden diferir un poco (revisión del código).
- **Avisos de revisión pendientes** (no bloqueantes): faltan tests para varios caminos de error del
  runner (horizonte o columnas distintas entre `fit` y `predict`, historial corto), el script de Optuna
  captura cualquier `ValueError` como poda y hay una división por cero posible si el MAE de Naive fuera
  cero en una ventana de ajuste.
- **No se probó la LSTM pura** (`nsmt`, de otro integrante): falta aislar qué aporta cada parte.

**Siguientes pasos:** terminar la validación con `--folds`, completar la tabla de la sección 6 y
cerrar los avisos de revisión con un commit de tests. La decisión de reemplazar o conservar el TCN
queda con el equipo (se recomienda conservarlo como referencia).

## 8. Cómo reproducir

```bash
uv sync
# Genera el parquet (una vez)
uv run python scripts/run_model.py booster --write-dataset
# Un corte contra Naive
uv run python scripts/run_model.py cnn_lstm --against naive
# 24 cortes (salidas en data/predictions/cnn_lstm/backtest/naive/)
uv run python scripts/run_model.py cnn_lstm --against naive --folds
```

Para usar GPU en Colab: [`colab_folds.ipynb`](colab_folds.ipynb) (cambia la variable
`MODEL`). Los resultados en GPU pueden variar un poco respecto a CPU porque el runner no activa
las opciones de determinismo de cuDNN.